# Experiment 08 — MLP with Concatenated Perch Scores + Mixup

**Builds on:** Exp07 (MLP OOF AUC 0.9238, cmAP 0.9112; best blend 40% MLP + 60% raw Perch).

**Motivation:** Exp07's best blend is 60% raw Perch — the MLP alone doesn't capture all the Perch signal. Two fixes:
1. **Concatenate raw Perch scores (234-dim) to the embedding input (1536-dim) → 1770-dim total.** The MLP can now learn to non-linearly refine Perch predictions using embedding context, instead of relying on a post-hoc linear blend.
2. **Mixup augmentation** (Beta(0.4,0.4)) in the training loop. Interpolates two examples (features + soft labels), acting as strong regularization on the small 739-sample dataset.

**Architecture:** `1770 → 512 → BN → ReLU → Drop(0.35) → 256 → BN → ReLU → Drop(0.35) → 234`

Expect the optimal raw-Perch blend to shift toward 0 (pure MLP) since Perch scores are already in the input.

In [1]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.special import expit as sigmoid
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
torch.manual_seed(42)
DEVICE = 'cpu'

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
CACHE_DIR = DATA_DIR / 'cache'

EMB_CACHE = CACHE_DIR / 'exp02_embeddings.npy'
LOG_CACHE = CACHE_DIR / 'exp02_logits.npy'
RESULTS_CSV = CACHE_DIR / 'exp08_mlp_concat_mixup_results.csv'
RESULTS_JSON = CACHE_DIR / 'exp08_mlp_concat_mixup_best.json'

N_SPLITS = 5
PRIOR_BLEND_GRID = [0.00, 0.025, 0.05, 0.10, 0.20]
PROBE_BLEND_GRID = [0.00, 0.10, 0.20, 0.30, 0.40, 0.60]  # expect lower optimal blend vs exp07

# MLP config
HIDDEN = [512, 256]
DROPOUT = 0.35
LR = 3e-4
WEIGHT_DECAY = 1e-3
MAX_EPOCHS = 300
PATIENCE = 25
BATCH_SIZE = 64
POS_WEIGHT_CAP = 30.0
MIXUP_ALPHA = 0.4

print('Project root:', PROJECT_ROOT)
print('PyTorch:', torch.__version__)

Project root: /Users/jroessler/Development/kaggle/birdclef
PyTorch: 2.2.2


In [2]:
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')
N_CLASSES = len(taxonomy)
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)
taxon_ids = taxonomy['primary_label'].tolist()
taxon_to_idx = {tid: i for i, tid in enumerate(taxon_ids)}

perch_labels = pd.read_csv(DATA_DIR / 'models/perch_tf/assets/labels.csv', header=0)
perch_labels.columns = ['scientific_name']
perch_labels['bc_index'] = np.arange(len(perch_labels))

mapping = taxonomy.merge(perch_labels, on='scientific_name', how='left')
comp_to_bc = {
    taxon_to_idx[str(row['primary_label'])]: int(row['bc_index'])
    for _, row in mapping[mapping['bc_index'].notna()].iterrows()
}

def time_to_sec(t):
    h, m, s = map(int, t.split(':'))
    return h * 3600 + m * 60 + s

def parse_soundscape_filename(filename):
    m = re.search(r'BC2026_Train_\d+_(S\d+)_([0-9]{8})_([0-9]{6})\.ogg', filename)
    if not m:
        return pd.Series({'site': 'UNK', 'hour_utc': -1, 'month': -1})
    site, ymd, hms = m.groups()
    return pd.Series({'site': site, 'hour_utc': int(hms[:2]), 'month': int(ymd[4:6])})

raw_df = pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')
raw_df['start_sec'] = raw_df['start'].apply(time_to_sec)
labels_df = (
    raw_df
    .drop_duplicates(subset=['filename', 'start_sec'])
    .sort_values(['filename', 'start_sec'])
    .reset_index(drop=True)
)
labels_df = pd.concat([labels_df, labels_df['filename'].apply(parse_soundscape_filename)], axis=1)

def parse_labels(label_str):
    vec = np.zeros(N_CLASSES, dtype=np.float32)
    for tid in str(label_str).split(';'):
        tid = tid.strip()
        if tid in taxon_to_idx:
            vec[taxon_to_idx[tid]] = 1.0
    return vec

Y = np.stack(labels_df['primary_label'].apply(parse_labels).values)
emb_full = np.load(EMB_CACHE).astype(np.float32)
logits_full = np.load(LOG_CACHE).astype(np.float32)

groups = labels_df['filename'].values
folds = list(GroupKFold(n_splits=N_SPLITS).split(emb_full, groups=groups))

print(f'Labels: {Y.shape}, files: {labels_df["filename"].nunique()}')

Labels: (739, 234), files: 66


In [3]:
def build_scores_with_proxy():
    scores = np.zeros((len(labels_df), N_CLASSES), dtype=np.float32)
    for comp_idx, bc_idx in comp_to_bc.items():
        scores[:, comp_idx] = logits_full[:, bc_idx]
    unmapped = mapping[mapping['bc_index'].isna()].copy()
    for _, row in unmapped.iterrows():
        target = str(row['primary_label'])
        genus = str(row['scientific_name']).split()[0]
        hits = perch_labels[
            perch_labels['scientific_name'].astype(str).str.match(rf'^{re.escape(genus)}\s', na=False)
        ]
        if hits.empty:
            continue
        comp_idx = taxon_to_idx[target]
        scores[:, comp_idx] = logits_full[:, hits['bc_index'].astype(int).values].max(axis=1)
    return scores

scores = build_scores_with_proxy()
raw_probs = sigmoid(scores)
print('Scores shape:', scores.shape)

Scores shape: (739, 234)


In [4]:
class MultiTaskMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return float(np.mean(aps))


def macro_auc(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        pos = y_true[:, c].sum()
        if 0 < pos < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return float(np.mean(aucs)), len(aucs)


def mixup_batch(x, y, alpha):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(len(x))
    return lam * x + (1.0 - lam) * x[idx], lam * y + (1.0 - lam) * y[idx]


def train_mlp_fold(X_tr, Y_tr, X_va, Y_va):
    in_dim = X_tr.shape[1]
    model = MultiTaskMLP(in_dim, HIDDEN, N_CLASSES, DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=10, min_lr=1e-5
    )

    pos_counts = Y_tr.sum(axis=0).clip(1)
    neg_counts = len(Y_tr) - pos_counts
    pw = np.clip(neg_counts / pos_counts, 1.0, POS_WEIGHT_CAP).astype(np.float32)
    pos_weight = torch.tensor(pw, device=DEVICE)
    active_mask = torch.tensor(Y_tr.sum(axis=0) > 0, device=DEVICE)

    X_tr_t = torch.tensor(X_tr, device=DEVICE)
    Y_tr_t = torch.tensor(Y_tr, device=DEVICE)
    X_va_t = torch.tensor(X_va, device=DEVICE)

    n = len(X_tr_t)
    best_val = -1.0
    best_pred = None
    wait = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(n)
        for start in range(0, n, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            xb, yb = mixup_batch(X_tr_t[idx], Y_tr_t[idx], MIXUP_ALPHA)
            logit_out = model(xb)
            loss = F.binary_cross_entropy_with_logits(
                logit_out[:, active_mask],
                yb[:, active_mask],
                pos_weight=pos_weight[active_mask],
                reduction='mean',
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_va_t).numpy()
        val_pred = sigmoid(val_logits)
        val_score = padded_cmap(Y_va, val_pred)
        scheduler.step(val_score)

        if val_score > best_val:
            best_val = val_score
            best_pred = val_pred.copy()
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    return best_pred, best_val, epoch + 1

print('Model and training functions defined.')

Model and training functions defined.


In [5]:
def fit_prior_tables(meta_df, y_train, strength_site=8.0, strength_hour=8.0, strength_site_hour=4.0):
    global_p = y_train.mean(axis=0).astype(np.float32)
    tables = {'global_p': global_p, 'site': {}, 'hour': {}, 'site_hour': {}}
    for site, idx in meta_df.groupby('site').groups.items():
        idx = np.array(list(idx), dtype=int)
        p = (y_train[idx].sum(axis=0) + strength_site * global_p) / (len(idx) + strength_site)
        tables['site'][str(site)] = p.astype(np.float32)
    for hour, idx in meta_df.groupby('hour_utc').groups.items():
        idx = np.array(list(idx), dtype=int)
        p = (y_train[idx].sum(axis=0) + strength_hour * global_p) / (len(idx) + strength_hour)
        tables['hour'][int(hour)] = p.astype(np.float32)
    for (site, hour), idx in meta_df.groupby(['site', 'hour_utc']).groups.items():
        idx = np.array(list(idx), dtype=int)
        p = (y_train[idx].sum(axis=0) + strength_site_hour * global_p) / (len(idx) + strength_site_hour)
        tables['site_hour'][(str(site), int(hour))] = p.astype(np.float32)
    return tables


def prior_probs_from_tables(meta_df, tables):
    out = np.repeat(tables['global_p'][None, :], len(meta_df), axis=0).astype(np.float32)
    for i, row in enumerate(meta_df.itertuples(index=False)):
        site = str(row.site)
        hour = int(row.hour_utc)
        if hour in tables['hour']:
            out[i] = 0.50 * out[i] + 0.50 * tables['hour'][hour]
        if site in tables['site']:
            out[i] = 0.50 * out[i] + 0.50 * tables['site'][site]
        if (site, hour) in tables['site_hour']:
            out[i] = 0.35 * out[i] + 0.65 * tables['site_hour'][(site, hour)]
    return np.clip(out, 1e-5, 1.0 - 1e-5)


def evaluate(name, y_pred, extra=None):
    auc, n_auc = macro_auc(Y, y_pred)
    row = {'method': name, 'macro_auc': auc, 'auc_classes': n_auc, 'padded_cmap': padded_cmap(Y, y_pred)}
    if extra:
        row.update(extra)
    return row

In [6]:
# Build input: scaled embeddings (1536) + scaled raw scores (234) = 1770 features
oof_mlp = np.zeros_like(raw_probs)
oof_prior = np.zeros_like(raw_probs)

for fold, (tr, va) in enumerate(folds):
    emb_scaler = StandardScaler()
    score_scaler = StandardScaler()

    emb_tr = emb_scaler.fit_transform(emb_full[tr])
    emb_va = emb_scaler.transform(emb_full[va])
    sc_tr = score_scaler.fit_transform(scores[tr])
    sc_va = score_scaler.transform(scores[va])

    X_tr = np.concatenate([emb_tr, sc_tr], axis=1).astype(np.float32)  # (n_tr, 1770)
    X_va = np.concatenate([emb_va, sc_va], axis=1).astype(np.float32)

    pred, best_val, stopped_epoch = train_mlp_fold(X_tr, Y[tr], X_va, Y[va])
    oof_mlp[va] = pred

    train_meta = labels_df.iloc[tr][['site', 'hour_utc']].reset_index(drop=True)
    val_meta = labels_df.iloc[va][['site', 'hour_utc']].reset_index(drop=True)
    tables = fit_prior_tables(train_meta, Y[tr])
    oof_prior[va] = prior_probs_from_tables(val_meta, tables)

    fold_auc, _ = macro_auc(Y[va], pred)
    fold_cmap = padded_cmap(Y[va], pred)
    print(f'Fold {fold+1}: best_val_cmap={best_val:.4f} epoch={stopped_epoch} '
          f'AUC={fold_auc:.4f} cmAP={fold_cmap:.4f}')

oof_auc, n_auc = macro_auc(Y, oof_mlp)
oof_cmap = padded_cmap(Y, oof_mlp)
print(f'\nOOF MLP (pure): AUC={oof_auc:.4f} cmAP={oof_cmap:.4f} (over {n_auc} classes)')

Fold 1: best_val_cmap=0.9790 epoch=36 AUC=0.8509 cmAP=0.9790


Fold 2: best_val_cmap=0.9758 epoch=46 AUC=0.8712 cmAP=0.9758


Fold 3: best_val_cmap=0.9797 epoch=72 AUC=0.9355 cmAP=0.9797


Fold 4: best_val_cmap=0.9541 epoch=73 AUC=0.7825 cmAP=0.9541


Fold 5: best_val_cmap=0.9673 epoch=50 AUC=0.8568 cmAP=0.9673



OOF MLP (pure): AUC=0.8747 cmAP=0.8781 (over 75 classes)


In [7]:
rows = []
rows.append(evaluate('raw_perch', raw_probs, {'probe_blend': 1.0, 'prior_blend': 0.0}))
rows.append(evaluate('mlp_pure', oof_mlp, {'probe_blend': 0.0, 'prior_blend': 0.0}))

for pb in PROBE_BLEND_GRID:
    blended = (1.0 - pb) * oof_mlp + pb * raw_probs
    rows.append(evaluate('mlp_raw_blend', blended, {'probe_blend': pb, 'prior_blend': 0.0}))
    for prior_b in PRIOR_BLEND_GRID:
        if prior_b == 0.0:
            continue
        final = (1.0 - prior_b) * blended + prior_b * oof_prior
        rows.append(evaluate('mlp_raw_prior', final, {'probe_blend': pb, 'prior_blend': prior_b}))

results = pd.DataFrame(rows).sort_values(['padded_cmap', 'macro_auc'], ascending=False).reset_index(drop=True)
results.to_csv(RESULTS_CSV, index=False)

print('Top 15 configurations:')
print(results.head(15).to_string(index=False, float_format=lambda x: f'{x:.4f}'))

best = results.iloc[0].to_dict()
best['exp07_baseline'] = {'macro_auc': 0.9238, 'padded_cmap': 0.9112}
with open(RESULTS_JSON, 'w') as f:
    json.dump(best, f, indent=2)
print('\nSaved:', RESULTS_JSON)

Top 15 configurations:
       method  macro_auc  auc_classes  padded_cmap  probe_blend  prior_blend
mlp_raw_prior     0.9086           75       0.9035       0.6000       0.0250
mlp_raw_blend     0.9078           75       0.9035       0.6000       0.0000
mlp_raw_prior     0.9092           75       0.9034       0.6000       0.0500
mlp_raw_prior     0.9098           75       0.9031       0.6000       0.1000
mlp_raw_prior     0.9091           75       0.9021       0.6000       0.2000
mlp_raw_prior     0.9076           75       0.9005       0.4000       0.0250
mlp_raw_blend     0.9071           75       0.9005       0.4000       0.0000
mlp_raw_prior     0.9081           75       0.9005       0.4000       0.0500
mlp_raw_prior     0.9088           75       0.9002       0.4000       0.1000
mlp_raw_prior     0.9087           75       0.8990       0.4000       0.2000
mlp_raw_blend     0.9053           75       0.8977       0.3000       0.0000
mlp_raw_prior     0.9058           75       0.8977   

## Result Notes
Exp07 baseline: OOF AUC 0.9238, padded cmAP 0.9112 (best blend 40% MLP + 60% raw Perch, prior 0.05).